In [ ]:
import itertools
import numpy as np
import pandas as pd
# 후보 설정
# 압력 : 4.6 bar ~ 5.4 bar까지 0.05 단위로 쪼갠 수치
# 유지시간 : 56 ~ 64 까지 0.5 단위로 쪼갠 수치
GRID = {
    'press_1': np.round(np.arange(4.6, 5.41, 0.05), 2),   # 설비 허용 범위
    'time_1' : np.arange(56, 64.1, 0.5),
}
SV = 10
# GRID

{'press_1': array([4.6 , 4.65, 4.7 , 4.75, 4.8 , 4.85, 4.9 , 4.95, 5.  , 5.05, 5.1 ,
        5.15, 5.2 , 5.25, 5.3 , 5.35, 5.4 ]),
 'time_1': array([56. , 56.5, 57. , 57.5, 58. , 58.5, 59. , 59.5, 60. , 60.5, 61. ,
        61.5, 62. , 62.5, 63. , 63.5, 64. ])}

In [25]:
from joblib import load
from sklearn.model_selection import train_test_split

# 2-5 단계에서 추출한 위험근접판정
risk = pd.read_csv('/content/data/risk.csv')

# 학습 모델
best = load('/content/data/best.joblib')

# 테스트할 데이터
X = pd.read_csv('/content/data/X_features.csv')
y = pd.read_csv('/content/data/y_target.csv').squeeze()
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [26]:
def recommend(x_row, model):
    cands = []

    # 압력과 유지시간을 모든 경우의 수로 짝을 지어 하나씩 꺼내 반복문을 돌린다.
    for p, t in itertools.product(GRID['press_1'], GRID['time_1']):
        # 받아온 row 를 복사하여 x 에 담는다.(원본보존)
        x = x_row.copy()
        # x['press_1'], x['time_1'] = p, t: 압력과 유지시간을 가상 후보값 (p, t)로 대입
        x['press_1'], x['time_1'] = p, t
        # 유지시간이 짧고나 길수록 치수 편차가 커진다.(그래서 이에 대한 내용을 추가)
        x['time_dev2'] = (t - 60) ** 2   
        # 데이터상 압력은(4.6 ~ 5.4 bar) 안에서는 압력이 올라갈수록 치수가 일관되게 줄어드는 선형적(직선적) 경향이 있어서 아래 코드를 굳이 넣을 필요가 없다.    
        # x['press_dev2'] = (p - 60) ** 2
           
        cands.append(x)
    cands = pd.DataFrame(cands)

    # 예측하고, SV 와의 절대값 차이를 계산하여 gap 컬럼 에 추가
    cands['pred'] = model.predict(cands[X.columns])
    cands['gap'] = (cands['pred'] - SV).abs()    
  
    p_change = (cands['press_1'] - x_row['press_1']).abs() # 압력의 변화량
    t_change = (cands['time_1'] - x_row['time_1']).abs() # 시간의 변화량
    # 서로 단위가 다르므로 0.2 bar 변화와 2초 변화를 동등한 1단위로 맞추어(정규화) 더해 준다.
    cands['change'] = (p_change/0.2) + (t_change/2)

    # 변경량이 작은 안을 우선 (현장 수용성 - 현장에서도 많이 바꾸는거 안좋아함)
    # 차이와 변화량이 작은 것을 우선적으로 정렬
    return cands.sort_values(['gap', 'change']).head(3)

In [53]:
risk[risk['grade'] == '위험']

,pred,actual,proximity,out_band,grade
73,9.951326,9.946295,0.973473,True,위험


In [54]:
#  SV 10.00mm, LSL 9.95mm, USL 10.05mm
for idx in risk[risk['grade'] == '위험'].index[:3]:
    print(f"Lot {idx}(위험등급 판정) - 예상수치: {risk.loc[idx, 'pred']:.4f}mm")
    # print(recommend(x_test.iloc[idx],best))
    print(recommend(x_test.iloc[idx], best)[['press_1', 'time_1', 'pred', 'gap']].to_string(index=False))

Lot 73(위험등급 판정) - 예상수치: 9.9513mm
 press_1  time_1     pred      gap
    4.65    64.0 9.985416 0.014584
    4.60    64.0 9.985360 0.014640
    4.70    64.0 9.985057 0.014943
